1. Filtrovanie vstupného katalógu


Táto časť spracovania načíta pôvodný morfologický katalóg, vyberie z neho galaxie s dostatočne vysokou pravdepodobnosťou orientácie disku „edge‑on“ a uloží zredukovaný dataset pre ďalšie kroky pipeline. Filtrácia prebieha podľa atribútu disk-edge-on_yes_fraction, pričom sa ponechávajú iba objekty s hodnotou ≥ 0.8.

In [ ]:
import pandas as pd

# Načítanie pôvodného morfologického katalógu
df = pd.read_csv("morphology_catalogue.csv")

# Názov stĺpca, podľa ktorého sa filtruje (pravdepodobnosť edge-on orientácie)
column_name = "disk-edge-on_yes_fraction"

# Výber iba tých galaxií, ktoré spĺňajú prahovú hodnotu
filtered_df = df[df[column_name] >= 0.8]

# Uloženie zredukovaného datasetu pre ďalšie spracovanie
filtered_df.to_csv("modified_dataset.csv", index=False)

print("Hotovo")
print(f"Pôvodný počet: {len(df)}")
print(f"Po filtrácii: {len(filtered_df)}")

2. Generovanie FITS výrezov (cutoutov)

V tejto fáze sa pre každú galaxiu v zredukovanom katalógu načíta príslušný FITS súbor, vypočíta sa poloha objektu v pixelových súradniciach pomocou WCS transformácie a vytvorí sa výrez (cutout) s definovanou uhlovou veľkosťou. Výstupom sú nové FITS súbory obsahujúce iba lokálnu oblasť okolo galaxie.

In [ ]:
from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata.utils import Cutout2D
import astropy.units as u
import pandas as pd
import os

# Dataset s vyfiltrovanými galaxiami
dataset_file = "modified_dataset.csv"
df = pd.read_csv(dataset_file)

# Vstupný priečinok s pôvodnými FITS súbormi
fits_folder = "FITS_vis_loc"

# Výstupný priečinok pre cutout FITS súbory
output_folder = "FITS_VIS_LOC_EXTRACTED"
os.makedirs(output_folder, exist_ok=True)

# Spracovanie každého riadku datasetu
for idx, row in df.iterrows():
    print(f"Spracovávam riadok {idx}...")

    # Názov FITS súboru pre danú galaxiu
    fits_file_name = row['vis_loc']
    fits_path = os.path.join(fits_folder, fits_file_name)

    # Načítanie FITS a WCS informácií
    hdu = fits.open(fits_path)[0]
    wcs = WCS(hdu.header)
    data = hdu.data

    # Svetové súradnice objektu
    ra = row['right_ascension']
    dec = row['declination']

    # Uhlová veľkosť cutoutu (zmenšená podľa potreby)
    cutout_width_arcsec = row['cutout_width_arcsec'] / 10

    # Prevod RA/DEC → pixelové súradnice
    x, y = wcs.world_to_pixel_values(ra, dec)

    # Definovanie veľkosti výrezu
    size = cutout_width_arcsec * u.arcsec

    # Vytvorenie cutoutu
    cutout = Cutout2D(data, (x, y), size, wcs=wcs)

    # Výstupný názov podľa stĺpca jpg_loc_gz_arcsinh_vis_only
    output_name = row['jpg_loc_gz_arcsinh_vis_only']
    output_path = os.path.join(output_folder, output_name)

    # Uloženie cutoutu do FITS
    hdu_cutout = fits.PrimaryHDU(data=cutout.data, header=cutout.wcs.to_header())
    hdu_cutout.writeto(output_path, overwrite=True)

    print(f"Uložené: {output_path}")

3. Generovanie log‑stretch vizualizácií

Táto časť pipeline prevádza FITS výrezy na vizuálne prehľadné JPG obrázky pomocou logaritmického kontrastného rozťahu. Používa sa knižnica APLpy, ktorá umožňuje jednoduché vykresľovanie FITS dát. Výstupné obrázky slúžia ako vstup pre následnú tvorbu segmentačných masiek.

In [ ]:
import os
import sys
import warnings
import gc
from astropy.utils.exceptions import AstropyDeprecationWarning
from astropy.io import fits
import aplpy
import matplotlib

# Použitie backendu bez GUI (nutné pri dávkovom spracovaní)
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm

# Potlačenie zastaraných Astropy varovaní
warnings.filterwarnings('ignore', category=AstropyDeprecationWarning)

# Vstupný priečinok s FITS cutoutmi
input_folder  = 'FITS_VIS_LOC_EXTRACTED'

# Výstupný priečinok pre log-stretch JPG obrázky
output_folder = 'LOG_STRETCH'
os.makedirs(output_folder, exist_ok=True)

# Zoznam FITS súborov na spracovanie
fits_files = sorted(f for f in os.listdir(input_folder) if f.endswith('.fits'))

# Hlavná slučka s progress barom
with tqdm(fits_files,
          desc="VYTVORENIE JPG Z FITS PRE LOG KONTRAST",
          unit="súbor",
          dynamic_ncols=True,
          file=sys.stdout) as pbar:

    for i, fits_file in enumerate(pbar):

        in_path  = os.path.join(input_folder, fits_file)
        out_path = os.path.join(
            output_folder,
            os.path.splitext(fits_file)[0] + ".jpg"
        )

        try:
            # Načítanie FITS súboru
            with fits.open(in_path, memmap=False) as hdul:

                # Príprava obrázkového plátna
                fig = plt.figure(figsize=(5, 5))

                # APLpy objekt pre vizualizáciu FITS dát
                f = aplpy.FITSFigure(hdul[0],
                                     figure=fig,
                                     auto_refresh=False)

                # Logaritmický kontrastný stretch
                f.show_grayscale(invert=False,
                                 stretch='log',
                                 vmin=0.001,
                                 vmax=0.1)

                # Skrytie osí, tickov a rámu pre čistý výstup
                f.axis_labels.hide()
                f.tick_labels.hide()
                f.ticks.hide()
                f.frame.set_linewidth(0)

                # Uloženie výsledného JPG
                plt.savefig(out_path,
                            dpi=150,
                            bbox_inches='tight',
                            pad_inches=0)

                # Uvoľnenie objektov
                f.close()
                plt.close(fig)
                del f, fig

        except Exception:
            # Ak je FITS poškodený alebo nečitateľný, preskočí sa
            pass

        # Priebežné čistenie pamäte každých 20 súborov
        if (i + 1) % 20 == 0:
            gc.collect()

print(f"\n Všetky obrázky boli úspešne uložené do priečinka: {output_folder}")

4. Generovanie segmentačných masiek z log‑stretch obrázkov

V tejto časti sa z log‑stretch JPG obrázkov vytvárajú binárne masky pomocou morfologických operácií. Pipeline zahŕňa logaritmický kontrastný stretch, gaussovské vyhladenie, rekonštrukciu podľa maxima, prahovanie podľa percentilov a následné čistenie masky (odstránenie malých objektov, dier a výber najväčšej súvislej oblasti).

In [ ]:
import os
import sys
import numpy as np
from skimage.io import imread, imsave
from skimage.util import img_as_float, img_as_ubyte
from skimage.filters import gaussian
from skimage.morphology import (
    reconstruction,
    remove_small_objects,
    remove_small_holes,
    binary_closing,
    disk
)
from skimage.measure import label, regionprops
from scipy.ndimage import binary_fill_holes
from skimage.exposure import adjust_log
from tqdm import tqdm

# -----------------------------------------------------
# Funkcia na extrakciu masky z log-stretch obrázka
# -----------------------------------------------------
def extract_log_mask(img_float, smooth_sigma=2.0,
                     high_percentile=98, low_percentile=85,
                     min_obj_size=500, hole_size=500,
                     final_close_disk=3):

    # Prevod na float [0..1]
    img = img_as_float(img_float)

    # Logaritmický kontrastný stretch
    log_stretched = adjust_log(img, gain=1)

    # Vyhladenie pre potlačenie šumu
    smoothed = gaussian(log_stretched, sigma=smooth_sigma)

    # Percentilové prahy
    hi = np.percentile(smoothed, high_percentile)
    lo = np.percentile(smoothed, low_percentile)

    # Seed pre rekonštrukciu podľa maxima
    seed = smoothed.copy()
    seed[seed < hi] = 0.0

    # Rekonštrukcia (morphological reconstruction)
    rec = reconstruction(seed, smoothed, method='dilation')

    # Prahovanie
    mask = rec >= lo

    # Odstránenie malých objektov a dier
    mask = remove_small_objects(mask, min_size=min_obj_size)
    mask = remove_small_holes(mask, area_threshold=hole_size)

    # Výber najväčšej súvislej oblasti (galaxia)
    lbl = label(mask)
    if lbl.max() > 0:
        props = regionprops(lbl)
        largest = max(props, key=lambda r: r.area)
        mask = (lbl == largest.label)

    # Záverečné morfologické uzatvorenie
    mask = binary_closing(mask, disk(final_close_disk))

    return img_as_ubyte(mask)


# -----------------------------------------------------
# Hlavná slučka spracovania všetkých JPG obrázkov
# -----------------------------------------------------
if __name__ == "__main__":
    input_folder  = 'LOG_STRETCH'
    output_folder = 'LOG_STRETCH_MASKS'
    os.makedirs(output_folder, exist_ok=True)

    # Zoradenie vstupných súborov
    jpg_files = sorted(
        f for f in os.listdir(input_folder)
        if f.lower().endswith('.jpg')
    )

    # Spracovanie s progress barom
    for fname in tqdm(
        jpg_files,
        desc="VYTVORENIE LOG-STRETCH MASIEK",
        unit="súbor",
        dynamic_ncols=True,
        file=sys.stdout
    ):
        in_path  = os.path.join(input_folder, fname)
        out_path = os.path.join(output_folder, fname)

        # Načítanie obrázka ako grayscale float
        img = imread(in_path, as_gray=True)

        # Extrakcia masky
        mask = extract_log_mask(
            img,
            smooth_sigma=2.0,
            high_percentile=98,
            low_percentile=85,
            min_obj_size=500,
            hole_size=500,
            final_close_disk=3
        )

        # Uloženie binárnej masky
        imsave(out_path, mask)

    print(f"\n Všetky masky uložené do priečinka '{output_folder}'.")